<a href="https://colab.research.google.com/github/hunkim98/earth_science/blob/main/lecture/EPS210_Lab9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Harvard EPS-210 AI for Earth and Planetary Science

Instructor: Mostafa Mouasvi

# Lab 9: Foundation Models for Geospatial Segmentation

# Lab: Fine-Tuning Prithvi-EO-2.0 for Wildfire Burn Scar Mapping
## NASA/IBM's Geospatial Foundation Model Applied to Disaster Response


---

### Learning Objectives

1. Understand **geospatial foundation models** (GFMs) — large-scale Vision Transformers pre-trained on satellite imagery
2. Learn the architecture of **Prithvi-EO-2.0**: ViT with 3D spatiotemporal patch embeddings and masked autoencoder (MAE) pre-training
3. Fine-tune Prithvi-EO-2.0 for **burn scar segmentation** using NASA's HLS Burn Scars dataset via **TerraTorch**
4. Compare fine-tuned GFM vs training a U-Net from scratch on the same task
5. Investigate **few-shot learning**: how few labeled images does a GFM need vs a baseline model?
6. Visualize multispectral features and understand why SWIR bands are critical for fire mapping

### Core Reference

**Szwarcman, D., et al. (2024).** Prithvi-EO-2.0: A Versatile Multi-Temporal Foundation Model for Earth Observation Applications. *IEEE JSTARS*. [arXiv:2412.02732](https://arxiv.org/abs/2412.02732)  
**Model:** [huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-300M](https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-300M)  
**Toolkit:** [github.com/IBM/terratorch](https://github.com/IBM/terratorch)

---
## Part 1: Background — Prithvi-EO-2.0 and Geospatial Foundation Models

### 1.1 The Problem

Earth observation generates petabytes of satellite imagery annually, but labeled data remains scarce. Training task-specific models from scratch for each application (flood mapping, crop classification, burn scars, etc.) is expensive and wasteful.

### 1.2 Prithvi-EO-2.0

Prithvi-EO-2.0, jointly developed by **NASA**, **IBM**, and **Jülich Supercomputing Centre**, is a geospatial foundation model designed to be fine-tuned for diverse EO tasks:

| Feature | Details |
|---------|--------|
| **Architecture** | Vision Transformer (ViT) with 3D spatiotemporal patch embeddings |
| **Pre-training** | Masked Autoencoder (MAE) — reconstruct randomly masked patches |
| **Training data** | 4.2M tiles from NASA's Harmonized Landsat-Sentinel-2 (HLS) at 30m |
| **Bands** | Blue, Green, Red, Narrow NIR, SWIR-1, SWIR-2 (6 spectral bands) |
| **Model sizes** | 300M and 600M parameters |
| **Temporal support** | Multi-timestamp input via 3D patch embedding (T=1 to 4) |
| **Location awareness** | Optional temporal + location (TL) embeddings (lat/lon, day-of-year) |

### 1.3 Key Innovations

1. **3D Patch Embeddings**: A 3D convolutional layer divides input into cubes of size $(t, h, w)$ across time, height, and width — capturing temporal dynamics naturally
2. **3D Positional Encodings**: Separate sinusoidal encodings for time, height, and width, combined into a single 3D encoding
3. **Temporal + Location (TL) variant**: Encodes center latitude/longitude and day-of-year — so the model "knows" where and when the image was taken

### 1.4 TerraTorch

**TerraTorch** is an open-source toolkit built on PyTorch Lightning that simplifies fine-tuning geospatial FMs. It provides:
- Pre-built encoder-decoder architectures (UPerNet, FCN) with Prithvi backbones
- Standard dataset loaders for HLS burn scars, Sen1Floods11, crop classification
- Configuration-driven training with hyperparameter search

### 1.5 Why Burn Scar Mapping?

Wildfire burn scar detection is a critical disaster response application:
- **SWIR bands** (Short-Wave Infrared) are highly sensitive to burned vegetation
- Post-fire landscapes show characteristic spectral signatures in the SWIR
- The paper demonstrates Prithvi outperforms U-Net on this task, especially with limited labeled data
- The [HLS Burn Scars dataset](https://huggingface.co/datasets/ibm-nasa-geospatial/hls_burn_scars) contains ~800 labeled 512×512 chips from the continental US

---
## Part 2: Environment Setup

In [ ]:
# ============================================================
# @title Install TerraTorch and dependencies
# ============================================================
!pip install -q peft
!pip install -q huggingface_hub

import os
import sys

try:
    import terratorch
    print('TerraTorch imported successfully.')
except ImportError:
    print('Installing TerraTorch (this updates NumPy to 2.x)...')
    !pip install -q terratorch rasterio
    print('Restarting runtime to load new NumPy version...')
    os.kill(os.getpid(), 9)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import glob, time, copy, warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')

# Verify terratorch
import terratorch
print(f'TerraTorch installed and verified.')

import subprocess
from huggingface_hub import snapshot_download

---
## Part 3: Downloading the HLS Burn Scars Dataset

The **HLS Burn Scars** dataset from NASA/IBM contains:
- ~800 labeled tiles, 512×512 pixels, from the **continental US** (2018–2021)
- **6 HLS bands**: Blue, Green, Red, Narrow NIR, SWIR-1, SWIR-2
- **Binary labels**: 0 = unburned, 1 = burned, -1 = no data/clouds
- Derived from Monitoring Trends in Burn Severity (MTBS) fire perimeters
- Available on [Hugging Face](https://huggingface.co/datasets/ibm-nasa-geospatial/hls_burn_scars)

For this lab, we'll download a subset and prepare it for fine-tuning.

In [ ]:
# ============================================================
# Download HLS Burn Scars dataset from Hugging Face
# ============================================================
DATA_DIR = 'hls_burn_scars'
os.makedirs(DATA_DIR, exist_ok=True)

print('Downloading HLS Burn Scars dataset from Hugging Face...')
snapshot_download(
    repo_id='ibm-nasa-geospatial/hls_burn_scars',
    repo_type='dataset',
    local_dir=DATA_DIR,
    allow_patterns=['*.tif'],  # Only download GeoTIFFs
)

# ============================================================
# Load and explore the multispectral data
# ============================================================
import rasterio
import os
import glob
import numpy as np
from huggingface_hub import snapshot_download, list_repo_files
import tarfile

BAND_NAMES = ['Blue', 'Green', 'Red', 'NIR Narrow', 'SWIR-1', 'SWIR-2']
BAND_WAVELENGTHS = [0.48, 0.56, 0.66, 0.87, 1.61, 2.20]  # µm

DATA_DIR = 'hls_burn_scars'

# Check if data exists; if not, download/extract
all_tifs = sorted(glob.glob(f'{DATA_DIR}/**/*_merged.tif', recursive=True))

# If no merged TIFs found, try to download/extract
if not all_tifs:
    print("Data not found. Checking repository...")
    try:
        repo_files = list_repo_files(repo_id='ibm-nasa-geospatial/hls_burn_scars', repo_type='dataset')
        tar_files = [f for f in repo_files if f.endswith('.tar.gz')]

        if tar_files:
            print(f"Found archive {tar_files[0]}. Downloading and extracting...")
            snapshot_download(
                repo_id='ibm-nasa-geospatial/hls_burn_scars',
                repo_type='dataset',
                local_dir=DATA_DIR,
                allow_patterns=['*.tar.gz']
            )
            for archive in glob.glob(f'{DATA_DIR}/*.tar.gz'):
                print(f"Extracting {archive}...")
                with tarfile.open(archive) as tar:
                    tar.extractall(path=DATA_DIR)
        else:
             # Fallback to TIF download if no tar found (unlikely based on previous run)
             snapshot_download(
                repo_id='ibm-nasa-geospatial/hls_burn_scars',
                repo_type='dataset',
                local_dir=DATA_DIR,
                allow_patterns=['**/*.tif']
            )
    except Exception as e:
        print(f"Error downloading data: {e}")

# Re-scan for files (UPDATED PATTERNS)
all_tifs = sorted(glob.glob(f'{DATA_DIR}/**/*_merged.tif', recursive=True))
all_masks = sorted(glob.glob(f'{DATA_DIR}/**/*mask.tif', recursive=True))


def load_chip(img_path, mask_path):
    """Load a single HLS chip and its burn scar mask."""
    with rasterio.open(img_path) as src:
        img = src.read().astype(np.float32)  # (6, H, W)
    with rasterio.open(mask_path) as src:
        mask = src.read(1).astype(np.int64)  # (H, W)
    return img, mask


# Match images to masks by filename pattern
pairs = []
for img_path in all_tifs:
    base = os.path.basename(img_path).replace('_merged.tif', '')
    # Look for a mask file that contains the base identifier
    # This handles .mask.tif vs _mask.tif differences automatically
    mask_candidates = [m for m in all_masks if base in os.path.basename(m)]

    if mask_candidates:
        pairs.append((img_path, mask_candidates[0]))

print(f'Matched {len(pairs)} image-mask pairs')

if len(pairs) > 0:
    # Load first chip to inspect
    sample_img, sample_mask = load_chip(*pairs[0])
    print(f'\nImage shape: {sample_img.shape} (bands, H, W)')
    print(f'Mask shape:  {sample_mask.shape}')
    print(f'Mask values: {np.unique(sample_mask)}')
    print(f'\nBand statistics:')
    for i, name in enumerate(BAND_NAMES):
        band = sample_img[i]
        valid = band[band > -9999]  # exclude nodata
        if len(valid) > 0:
            print(f'  {name:12s} ({BAND_WAVELENGTHS[i]:.2f} µm): '
                  f'min={valid.min():.4f}, max={valid.max():.4f}, mean={valid.mean():.4f}')
else:
    print("Error: No image pairs found. Please check directory content:")
    print(glob.glob(f'{DATA_DIR}/**/*', recursive=True)[:20])

In [ ]:
# ============================================================
# Visualize burn scars — RGB + SWIR false color + mask
# ============================================================

def normalize_band(band, pmin=2, pmax=98):
    """Percentile-based normalization."""
    valid = band[band > -9999]
    if len(valid) == 0:
        return np.zeros_like(band)
    lo, hi = np.percentile(valid, pmin), np.percentile(valid, pmax)
    out = np.clip((band - lo) / (hi - lo + 1e-8), 0, 1)
    out[band <= -9999] = 0
    return out


def make_rgb(img):
    """True color composite (R, G, B)."""
    return np.stack([normalize_band(img[2]),
                     normalize_band(img[1]),
                     normalize_band(img[0])], axis=-1)


def make_swir_false(img):
    """False color: SWIR-2, NIR, Green — highlights burn scars in red/magenta."""
    return np.stack([normalize_band(img[5]),   # SWIR-2
                     normalize_band(img[3]),   # NIR
                     normalize_band(img[1])],  # Green
                    axis=-1)


# Show examples
if len(pairs) == 0:
    print("No images matched to visualize. Please check data loading.")
else:
    n_show = min(4, len(pairs))
    # squeeze=False ensures axes is always 2D array even if n_show=1
    fig, axes = plt.subplots(n_show, 4, figsize=(16, 4*n_show), squeeze=False)
    burn_cmap = ListedColormap(['#2c3e50', '#95a5a6', '#e74c3c'])

    for row in range(n_show):
        img, mask = load_chip(*pairs[row])
        burned_pct = 100 * (mask == 1).sum() / (mask >= 0).sum() if (mask >= 0).any() else 0

        axes[row, 0].imshow(make_rgb(img))
        axes[row, 0].set_title('True Color (RGB)', fontsize=10)
        axes[row, 0].axis('off')

        axes[row, 1].imshow(make_swir_false(img))
        axes[row, 1].set_title('SWIR False Color\n(SWIR2, NIR, Green)', fontsize=10)
        axes[row, 1].axis('off')

        axes[row, 2].imshow(mask, cmap=burn_cmap, vmin=-1, vmax=1, interpolation='nearest')
        axes[row, 2].set_title(f'Burn Scar Mask\n({burned_pct:.1f}% burned)', fontsize=10)
        axes[row, 2].axis('off')

        # Overlay: burn scar boundary on SWIR
        swir_vis = make_swir_false(img)
        overlay = swir_vis.copy()
        burn_boundary = np.zeros_like(mask, dtype=bool)
        from scipy.ndimage import binary_dilation
        if (mask == 1).any():
            dilated = binary_dilation(mask == 1, iterations=2)
            burn_boundary = dilated & ~(mask == 1)
            overlay[burn_boundary] = [1, 1, 0]  # yellow boundary
        axes[row, 3].imshow(overlay)
        axes[row, 3].set_title('SWIR + Burn Boundary', fontsize=10)
        axes[row, 3].axis('off')

    patches = [mpatches.Patch(color='#95a5a6', label='Unburned'),
               mpatches.Patch(color='#e74c3c', label='Burned'),
               mpatches.Patch(color='#2c3e50', label='No Data')]
    fig.legend(handles=patches, loc='lower center', ncol=3, fontsize=11)
    fig.suptitle('HLS Burn Scars Dataset — Multispectral Views of Wildfire Impacts',
                 fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0.05, 1, 0.96]); plt.show()

**Feature Validation and Exploratory Data Analysis (EDA)**

Here we try to understand why certain satellite bands (like SWIR) are more effective than others for identifying wildfire damage.

This step acts as a sanity check. If the separability index is low across all bands, no amount of complex architecture (like a Transformer or CNN) will perform well because the input data lacks a clear discriminative signal.

In [ ]:
# ============================================================
# Spectral signature: burned vs unburned pixels
# ============================================================

burned_spectra = []
unburned_spectra = []

# we loop through a subset of image chips and masks. For each chip, it isolates pixels labeled as "burned" (mask=1) and "unburned" (mask=0).
# We then calculate the mean reflectance across all available bands for these two categories.
for idx in range(min(20, len(pairs))):
    img, mask = load_chip(*pairs[idx])
    valid = (img[0] > -9999) & (mask >= 0)

    burned_px = valid & (mask == 1)
    unburned_px = valid & (mask == 0)
    # Thresholding: The if burned_px.sum() > 100 check ensures that the statistics are based on a representative sample size, preventing noise from skewed single-pixel outliers.
    if burned_px.sum() > 100:
        spec = img[:, burned_px].mean(axis=1)
        burned_spectra.append(spec)
    if unburned_px.sum() > 100:
        spec = img[:, unburned_px].mean(axis=1)
        unburned_spectra.append(spec)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) Mean spectra
if burned_spectra:
    b_mean = np.mean(burned_spectra, axis=0)
    b_std = np.std(burned_spectra, axis=0)
    axes[0].fill_between(BAND_WAVELENGTHS, b_mean-b_std, b_mean+b_std,
                         alpha=0.2, color='#e74c3c')
    axes[0].plot(BAND_WAVELENGTHS, b_mean, 'o-', color='#e74c3c',
                 linewidth=2, label='Burned')

if unburned_spectra:
    u_mean = np.mean(unburned_spectra, axis=0)
    u_std = np.std(unburned_spectra, axis=0)
    axes[0].fill_between(BAND_WAVELENGTHS, u_mean-u_std, u_mean+u_std,
                         alpha=0.2, color='#2ecc71')
    axes[0].plot(BAND_WAVELENGTHS, u_mean, 's-', color='#2ecc71',
                 linewidth=2, label='Unburned')

for i, name in enumerate(BAND_NAMES):
    axes[0].axvline(BAND_WAVELENGTHS[i], color='gray', alpha=0.2, linestyle='--')
    axes[0].annotate(name, (BAND_WAVELENGTHS[i], axes[0].get_ylim()[1]*0.95),
                     rotation=45, fontsize=7, ha='center')

axes[0].set_xlabel('Wavelength (µm)'); axes[0].set_ylabel('Reflectance')
axes[0].set_title('(a) Mean Spectral Signatures'); axes[0].legend()

# (b) Per-band separability
if burned_spectra and unburned_spectra:
    b_arr = np.array(burned_spectra)
    u_arr = np.array(unburned_spectra)
    separability = np.abs(b_arr.mean(0) - u_arr.mean(0)) / \
                   (np.sqrt(b_arr.std(0)**2 + u_arr.std(0)**2) + 1e-8)
    colors = ['#3498db' if s < 1 else '#e74c3c' for s in separability]
    axes[1].bar(BAND_NAMES, separability, color=colors, edgecolor='black', alpha=0.8)
    axes[1].set_ylabel('Separability Index (d\')'); axes[1].set_title('(b) Burn vs Unburn Separability')
    axes[1].axhline(1.0, color='gray', linestyle='--', alpha=0.5)
    axes[1].set_xticklabels(BAND_NAMES, rotation=30, ha='right')

fig.suptitle('Why SWIR Bands Matter for Burn Scar Detection',
             fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

a) Mean Spectral Signatures plot (left) shows the "shape" of the light reflected by the two surfaces across different wavelengths. Where the red (burned) and green (unburned) lines diverge most significantly is wavelength most useful for our purpose. The fill_between area (the shaded "cloud" around the lines) represents the variance. If the shaded areas for burned and unburned overlap heavily in a specific band, that band is a poor predictor.

b) Showes Separability Index (often called the Sensitivity Index or Cohen's $d$):$$d' = \frac{|\mu_{burned} - \mu_{unburned}|}{\sqrt{\sigma^2_{burned} + \sigma^2_{unburned}}}$$ It measures the distance between the means relative to the standard deviation.The Threshold: The code colors bars red if the separability is $> 1$. This tells you which specific bands (e.g., SWIR1, SWIR2) provide the "cleanest" signal for your model to learn from.

---
## Part 4: Preparing Data for Prithvi-EO-2.0

Prithvi-EO-2.0 expects input in the format `(B, C, T, H, W)` — batch, channels, timestamps, height, width. For single-timestamp burn scar mapping, we use `T=1`. The 6 HLS bands map directly to Prithvi's pre-training bands.

In [ ]:
# ============================================================
# PyTorch Dataset for HLS Burn Scars (Optimized with RAM Cache)
# ============================================================
import cv2  # For fast resizing
from tqdm.notebook import tqdm

class BurnScarDataset(Dataset):
    """
    HLS Burn Scars dataset for Prithvi-EO-2.0 fine-tuning.
    """
    # HLS band normalization (from Prithvi pre-training stats)
    MEANS = np.array([0.033349706741586264, 0.05701185520536176,
                      0.05889748132001316, 0.2323245113436119,
                      0.1972854853760658, 0.11944914225186566])
    STDS = np.array([0.02269135568823774, 0.026807560223070237,
                     0.04004109844362779, 0.07791732423672691,
                     0.08708738838140137, 0.07241979477437814])

    def __init__(self, pairs, crop_size=128, augment=False, preload=True):
        self.pairs = pairs
        self.crop_size = crop_size
        self.augment = augment
        self.preload = preload
        self.cache = [None] * len(pairs)

        if self.preload:
            print(f"Pre-loading {len(pairs)} images to RAM (resizing to 128x128 for speed)...")
            for i in tqdm(range(len(pairs)), desc="Caching"):
                self.cache[i] = self._load_and_process(i)

    def _load_and_process(self, idx):
        img, mask = load_chip(*self.pairs[idx])

        if img.shape[1] > 128:
            # Transpose to (H, W, C) for cv2
            img_t = img.transpose(1, 2, 0)
            img_t = cv2.resize(img_t, (128, 128), interpolation=cv2.INTER_LINEAR)
            img = img_t.transpose(2, 0, 1)

            # Resize mask (nearest neighbor to preserve class values)
            # IMPORTANT: -1 (ignore) becomes 255 in uint8. We must convert it back.
            mask = cv2.resize(mask.astype(np.uint8), (128, 128), interpolation=cv2.INTER_NEAREST).astype(np.int64)
            mask[mask == 255] = -1

        # Replace nodata with 0
        img[img <= -9999] = 0

        # Normalize with pre-training statistics
        for b in range(6):
            img[b] = (img[b] - self.MEANS[b]) / (self.STDS[b] + 1e-8)

        return img, mask

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        if self.preload and self.cache[idx] is not None:
            img, mask = self.cache[idx]
            img = img.copy() # Important for augmentation
            mask = mask.copy()
        else:
            img, mask = self._load_and_process(idx)

        # Random crop to crop_size
        H, W = img.shape[1], img.shape[2]
        if H >= self.crop_size and W >= self.crop_size:
            if self.augment:
                y = np.random.randint(0, H - self.crop_size + 1)
                x = np.random.randint(0, W - self.crop_size + 1)
            else: # Center crop
                y = (H - self.crop_size) // 2
                x = (W - self.crop_size) // 2
            img = img[:, y:y+self.crop_size, x:x+self.crop_size]
            mask = mask[y:y+self.crop_size, x:x+self.crop_size]
        else:
             # Safety pad if image < crop_size
             pad_h = max(0, self.crop_size - H)
             pad_w = max(0, self.crop_size - W)
             img = np.pad(img, ((0,0), (0, pad_h), (0, pad_w)))
             mask = np.pad(mask, ((0, pad_h), (0, pad_w)))
             img = img[:, :self.crop_size, :self.crop_size]
             mask = mask[:self.crop_size, :self.crop_size]

        # Augmentation: random flips
        if self.augment:
            if np.random.rand() > 0.5:
                img = img[:, ::-1, :].copy()
                mask = mask[::-1, :].copy()
            if np.random.rand() > 0.5:
                img = img[:, :, ::-1].copy()
                mask = mask[:, ::-1].copy()

        # Add temporal dimension: (C, H, W) -> (C, T=1, H, W)
        img = img[:, np.newaxis, :, :]  # (6, 1, 128, 128)

        return torch.FloatTensor(img), torch.LongTensor(mask)


# Split data: 70% train, 15% val, 15% test
np.random.seed(42)
np.random.shuffle(pairs)
n = len(pairs)
n_train = int(0.7 * n)
n_val = int(0.15 * n)

train_pairs = pairs[:n_train]
val_pairs = pairs[n_train:n_train+n_val]
test_pairs = pairs[n_train+n_val:]

print("Initializing datasets with RAM caching (128x128)...")
train_ds = BurnScarDataset(train_pairs, augment=True, preload=True)
val_ds = BurnScarDataset(val_pairs, preload=True)
test_ds = BurnScarDataset(test_pairs, preload=True)

train_dl = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=0, drop_last=True)
val_dl = DataLoader(val_ds, batch_size=4, num_workers=0)
test_dl = DataLoader(test_ds, batch_size=4, num_workers=0)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

---
## Part 5: Loading Prithvi-EO-2.0 and Building the Segmentation Model

We use TerraTorch's `EncoderDecoderFactory` to build a segmentation model with:
- **Backbone**: Prithvi-EO-2.0-300M (pre-trained ViT encoder, 300M params)
- **Neck**: Select multi-scale feature indices + reshape tokens to spatial feature maps
- **Decoder**: UPerNet — a standard dense prediction decoder that aggregates multi-scale features
- **Head**: 2-class output (unburned, burned)

In [ ]:
# ============================================================
# Build Prithvi-EO-2.0 segmentation model via TerraTorch
# ============================================================
from terratorch.models import EncoderDecoderFactory
from terratorch.datasets import HLSBands

model_factory = EncoderDecoderFactory()

prithvi_model = model_factory.build_model(
    task='segmentation',
    backbone='prithvi_eo_v2_300',
    backbone_pretrained=True,
    backbone_bands=[
        HLSBands.BLUE,
        HLSBands.GREEN,
        HLSBands.RED,
        HLSBands.NIR_NARROW,
        HLSBands.SWIR_1,
        HLSBands.SWIR_2,
    ],
    backbone_num_frames=1,  # single timestamp

    # Neck is how we connect the endoder to the decoder!
    # Prithvi-300M is made up of 24 sequential transformer layers (blocks 0 through 23).
    # Instead of only taking the output of the very last layer, 'SelectIndices' tells the model to pull out the features from the 6th, 12th, 18th, and 24th layers.
    necks=[
        {'name': 'SelectIndices', 'indices': [5, 11, 17, 23]},
        {'name': 'ReshapeTokensToImage'}, # This takes flat 1D sequence of transformer tokens and reshapes them back into their original 2D spatial grid layout so the decoder can process them like normal images.
    ],

    # UPerNet is a segmentation decoders that work best with "multi-scale" features.
    decoder='UperNetDecoder',
    decoder_channels=256,
    head_dropout=0.1,
    num_classes=2,
).to(device)

# Count parameters
total = sum(p.numel() for p in prithvi_model.parameters())
trainable = sum(p.numel() for p in prithvi_model.parameters() if p.requires_grad)
print(f'\nPrithvi-EO-2.0 Segmentation Model:')
print(f'  Total params:     {total/1e6:.1f}M')
print(f'  Trainable params: {trainable/1e6:.1f}M')

---
## Part 6: Fine-Tuning Prithvi-EO-2.0 on Burn Scars


In [ ]:
# ============================================================
# Fine-tuning strategy: freeze backbone, train decoder only
# ============================================================

def freeze_backbone(model):
    """Freeze the Prithvi encoder, train only decoder + head."""
    for name, param in model.named_parameters():
        if 'encoder' in name or 'backbone' in name:
            param.requires_grad = False
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'After freezing backbone: {trainable/1e6:.2f}M trainable '
          f'({100*trainable/total:.1f}% of {total/1e6:.1f}M)')

# Start with frozen backbone (paper approach for limited data)
freeze_backbone(prithvi_model)

In [ ]:
# ============================================================
# Configure LoRA (Low-Rank Adaptation)
# ============================================================
from peft import LoraConfig, get_peft_model, TaskType

def unfreeze_all(model):
    """Unfreeze all parameters for full fine-tuning."""
    for param in model.parameters():
        param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'All {trainable/1e6:.1f}M parameters unfrozen')

# 1. Unfreeze everything first to reset state (in case it was frozen before)
unfreeze_all(prithvi_model)

# 2. Define LoRA Config
# We target 'qkv' (Query/Key/Value) projections in the ViT attention blocks
lora_config = LoraConfig(
    r=8,                  # Rank
    lora_alpha=16,        # Alpha scaling
    target_modules=["qkv"], # Target modules in timm ViT
    lora_dropout=0.2,
    bias="none",
)

# 3. Wrap the model
# This freezes the base model but enables gradients for LoRA adapters
prithvi_lora = get_peft_model(prithvi_model, lora_config)

# 4. Unfreeze Decoder and Head
# PEFT freezes the entire base model (including decoder/head) except adapters.
# We must manually unfreeze the non-backbone parts (Neck, Decoder, Head).
for name, param in prithvi_lora.named_parameters():
    # The backbone is in 'encoder', everything else (neck, decoder, head) should be trained
    if "encoder" not in name:
        param.requires_grad = True

prithvi_lora.print_trainable_parameters()

In [ ]:
# ============================================================
# Fine-tune with LoRA
# ============================================================
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def compute_iou(pred, gt, cls=1):
    """Compute IoU for a single class."""
    pred_c = (pred == cls)
    gt_c = (gt == cls)
    valid = (gt >= 0)  # ignore -1
    pred_c = pred_c & valid
    gt_c = gt_c & valid
    inter = (pred_c & gt_c).sum()
    union = (pred_c | gt_c).sum()
    return inter / (union + 1e-8)

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        output = model(imgs)
        # TerraTorch models return a ModelOutput object
        logits = output.output if hasattr(output, 'output') else output
        # Resize logits to match mask if needed
        if logits.shape[-2:] != masks.shape[-2:]:
            logits = F.interpolate(logits, size=masks.shape[-2:],
                                   mode='bilinear', align_corners=False)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_ious = []
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            output = model(imgs)
            logits = output.output if hasattr(output, 'output') else output
            if logits.shape[-2:] != masks.shape[-2:]:
                logits = F.interpolate(logits, size=masks.shape[-2:],
                                       mode='bilinear', align_corners=False)
            total_loss += criterion(logits, masks).item()
            preds = logits.argmax(1).cpu().numpy()
            gts = masks.cpu().numpy()
            for p, g in zip(preds, gts):
                all_ious.append(compute_iou(p, g, cls=1))
    return total_loss / len(loader), np.mean(all_ious)

def train_model(model, train_dl, val_dl, epochs=20, lr=1e-4, label=''):
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss(ignore_index=-1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    hist = {'tl': [], 'vl': [], 'viou': []}
    best_iou = 0

    for ep in range(epochs):
        t0 = time.time()
        tl = train_epoch(model, train_dl, optimizer, criterion)
        vl, viou = evaluate(model, val_dl, criterion)
        scheduler.step()

        hist['tl'].append(tl); hist['vl'].append(vl); hist['viou'].append(viou)
        if viou > best_iou: best_iou = viou

        dt = time.time() - t0
        print(f'  [{label}] Ep {ep+1:2d}/{epochs} | Train: {tl:.4f} | '
              f'Val: {vl:.4f} | Burn IoU: {viou:.4f} | Best: {best_iou:.4f} | {dt:.1f}s')

    return hist, best_iou

print('Fine-tuning Prithvi-EO-2.0 (LoRA backbone + Full Decoder)...\n')

# We use the same training loop
hist_lora, best_lora = train_model(
    prithvi_lora, train_dl, val_dl, epochs=20, lr=1e-4, label='Prithvi-LoRA')

print(f'\nBest burn scar IoU (LoRA): {best_lora:.4f}')

In [ ]:
# ============================================================
# Plot Training History
# ============================================================
plt.figure(figsize=(12, 5))

# Loss Curve
plt.subplot(1, 2, 1)
plt.plot(hist_lora['tl'], label='Train Loss')
plt.plot(hist_lora['vl'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training & Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)

# IoU Curve
plt.subplot(1, 2, 2)
plt.plot(hist_lora['viou'], label='Val Burn IoU', color='green')
plt.xlabel('Epoch')
plt.ylabel('IoU')
plt.title('Validation Burn IoU')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Visualize Predictions
# ============================================================
def denormalize(img_tensor):
    """Convert normalized tensor back to numpy array for vis."""
    # (C, T, H, W) -> (C, H, W)
    img = img_tensor.squeeze(1).cpu().numpy()

    means = BurnScarDataset.MEANS[:, None, None]
    stds = BurnScarDataset.STDS[:, None, None]

    img = img * stds + means
    return img

# Get a batch from test set
imgs, masks = next(iter(test_dl))
imgs = imgs.to(device)

# Predict
prithvi_lora.eval()
with torch.no_grad():
    output = prithvi_lora(imgs)
    logits = output.output if hasattr(output, 'output') else output
    if logits.shape[-2:] != masks.shape[-2:]:
        logits = F.interpolate(logits, size=masks.shape[-2:],
                               mode='bilinear', align_corners=False)
    preds = logits.argmax(1).cpu().numpy()

# Visualize
fig, axes = plt.subplots(4, 4, figsize=(16, 16))

for i in range(4):
    # Prepare images
    img_np = denormalize(imgs[i])
    mask_np = masks[i].cpu().numpy()
    pred_np = preds[i]

    # RGB
    rgb = np.stack([normalize_band(img_np[2]),
                    normalize_band(img_np[1]),
                    normalize_band(img_np[0])], axis=-1)

    # SWIR False Color
    swir = np.stack([normalize_band(img_np[5]),
                     normalize_band(img_np[3]),
                     normalize_band(img_np[1])], axis=-1)

    # Plot
    axes[i, 0].imshow(rgb)
    axes[i, 0].set_title("Input RGB")
    axes[i, 0].axis('off')

    axes[i, 1].imshow(swir)
    axes[i, 1].set_title("Input SWIR (False Color)")
    axes[i, 1].axis('off')

    axes[i, 2].imshow(mask_np, vmin=-1, vmax=1, cmap=ListedColormap(['#95a5a6', '#2c3e50', '#e74c3c']), interpolation='nearest')
    axes[i, 2].set_title("Ground Truth")
    axes[i, 2].axis('off')

    axes[i, 3].imshow(pred_np, vmin=-1, vmax=1, cmap=ListedColormap(['#95a5a6', '#2c3e50', '#e74c3c']), interpolation='nearest')
    axes[i, 3].set_title("Prediction")
    axes[i, 3].axis('off')

plt.tight_layout()
plt.show()

---
## Part 7: Baseline — Training a U-Net from Scratch

To quantify the benefit of the pre-trained foundation model, we train a lightweight U-Net from scratch on the same data. This follows the paper's comparison methodology.

In [ ]:
# ============================================================
# Lightweight U-Net baseline (trained from scratch)
# ============================================================

class UNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(True))
    def forward(self, x): return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_ch=6, n_classes=2):
        super().__init__()
        self.enc1 = UNetBlock(in_ch, 64)
        self.enc2 = UNetBlock(64, 128)
        self.enc3 = UNetBlock(128, 256)
        self.enc4 = UNetBlock(256, 512)
        self.pool = nn.MaxPool2d(2)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec3 = UNetBlock(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec2 = UNetBlock(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec1 = UNetBlock(128, 64)
        self.head = nn.Conv2d(64, n_classes, 1)

    def forward(self, x):
        # x shape: (B, C=6, T=1, H, W) — squeeze temporal dim
        if x.ndim == 5:
            x = x.squeeze(2)  # (B, 6, H, W)
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(e4), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        return self.head(d1)

unet = UNet(in_ch=6, n_classes=2).to(device)
unet_params = sum(p.numel() for p in unet.parameters())
print(f'U-Net: {unet_params/1e6:.1f}M params (all trainable, from scratch)')

print('\nTraining U-Net from scratch...\n')
hist_unet, best_unet = train_model(unet, train_dl, val_dl, epochs=20, lr=1e-3, label='U-Net')
print(f'\nBest burn scar IoU: {best_unet:.4f}')

---
## Part 8: Prithvi vs U-Net — Head-to-Head Comparison

We compare the two models on the test set and visualize predictions side by side.

In [ ]:
# ============================================================
# Test set evaluation
# ============================================================
criterion = nn.CrossEntropyLoss(ignore_index=-1)

_, prithvi_test_iou = evaluate(prithvi_model, test_dl, criterion)
_, unet_test_iou = evaluate(unet, test_dl, criterion)

print(f'Test Set Burn Scar IoU:')
print(f'  Prithvi-EO-2.0 (frozen backbone): {prithvi_test_iou:.4f}')
print(f'  U-Net (from scratch):             {unet_test_iou:.4f}')
print(f'  Δ (Prithvi - UNet):               {prithvi_test_iou - unet_test_iou:+.4f}')

In [ ]:
# ============================================================
# Visual comparison on test images
# ============================================================
n_show = 4
fig, axes = plt.subplots(n_show, 4, figsize=(16, 4*n_show))

prithvi_model.eval(); unet.eval()
burn_cmap = ListedColormap(['#2c3e50', '#e74c3c'])

for row in range(min(n_show, len(test_pairs))):
    x, gt = test_ds[row]
    x_dev = x.unsqueeze(0).to(device)

    with torch.no_grad():
        p_out = prithvi_model(x_dev)
        p_logits = p_out.output if hasattr(p_out, 'output') else p_out
        if p_logits.shape[-2:] != gt.shape[-2:]:
            p_logits = F.interpolate(p_logits, size=gt.shape[-2:],
                                     mode='bilinear', align_corners=False)
        p_pred = p_logits.argmax(1).squeeze().cpu().numpy()

        u_logits = unet(x_dev)
        u_pred = u_logits.argmax(1).squeeze().cpu().numpy()

    gt_np = gt.numpy()
    # Denormalize for display
    img_np = x.squeeze(1).numpy()  # (6, H, W)
    img_rgb = np.stack([
        np.clip(img_np[2] * train_ds.STDS[2] + train_ds.MEANS[2], 0, 1),
        np.clip(img_np[1] * train_ds.STDS[1] + train_ds.MEANS[1], 0, 1),
        np.clip(img_np[0] * train_ds.STDS[0] + train_ds.MEANS[0], 0, 1)
    ], axis=-1)
    img_rgb = np.clip(img_rgb * 5, 0, 1)  # brightness boost

    p_iou = compute_iou(p_pred, gt_np)
    u_iou = compute_iou(u_pred, gt_np)

    axes[row,0].imshow(img_rgb); axes[row,0].set_title('Input RGB'); axes[row,0].axis('off')
    axes[row,1].imshow(gt_np, cmap=burn_cmap, vmin=0, vmax=1, interpolation='nearest')
    axes[row,1].set_title('Ground Truth'); axes[row,1].axis('off')
    axes[row,2].imshow(p_pred, cmap=burn_cmap, vmin=0, vmax=1, interpolation='nearest')
    axes[row,2].set_title(f'Prithvi  IoU={p_iou:.3f}'); axes[row,2].axis('off')
    axes[row,3].imshow(u_pred, cmap=burn_cmap, vmin=0, vmax=1, interpolation='nearest')
    axes[row,3].set_title(f'U-Net  IoU={u_iou:.3f}'); axes[row,3].axis('off')

patches = [mpatches.Patch(color='#2c3e50', label='Unburned'),
           mpatches.Patch(color='#e74c3c', label='Burned')]
fig.legend(handles=patches, loc='lower center', ncol=2, fontsize=11)
fig.suptitle('Burn Scar Detection: Prithvi-EO-2.0 vs U-Net (from Scratch)',
             fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0.04, 1, 0.96]); plt.show()

---
## Part 9: Exercises

### Exercise 1: LoRA Adaptation of Prithvi (⭐⭐)
Instead of freezing the backbone or fully fine-tuning, inject LoRA adapters (rank=4) into Prithvi's attention layers. Compare three strategies: frozen backbone, LoRA, and full fine-tuning. Plot IoU vs trainable parameters.

### Exercise 2: Multi-Temporal Burn Monitoring (⭐⭐)
Prithvi-EO-2.0 supports multi-timestamp input (T=1 to 4). Create a dataset with pre-fire and post-fire images for the same location. Fine-tune Prithvi with T=2 input (before + after). Does temporal information improve burn scar detection compared to single-timestamp?

### Exercise 3: Flood Mapping Transfer (⭐⭐)
Download the Sen1Floods11 dataset and fine-tune the same Prithvi model for flood detection instead of burn scars. How does performance compare? Can you initialize from the burn-scar-adapted weights (sequential fine-tuning)?

### Exercise 4: Decoder Architecture Study (⭐⭐)
The paper tested UPerNet, FCN, and custom decoders. Replace `UperNetDecoder` with `FCNDecoder` in TerraTorch and compare. Also try reducing `decoder_channels` from 256 to 64 — how much performance do you lose?

### Exercise 5: Prithvi-300M vs Prithvi-600M (⭐⭐⭐)
Load `prithvi_eo_v2_600` (600M parameters) and compare to the 300M version. Does the larger model improve burn scar IoU? At what training set size does the 600M model's advantage become significant? (Requires A100 or L4 GPU.)

### Exercise 6: Location-Aware Model (⭐⭐⭐)
The TL (Temporal-Location) variant of Prithvi-EO-2.0 encodes latitude/longitude and day-of-year. Load `prithvi_eo_v2_300_tl` and provide location metadata during fine-tuning. Does geographic awareness improve burn scar mapping, especially for fires in ecologically different regions?

### Exercise 7: GEO-Bench Evaluation (⭐⭐⭐)
Install `geo_bench` and evaluate your fine-tuned Prithvi model on the full GEO-Bench segmentation suite. Compare to the published results in Table 1 of the paper. Are there tasks where Prithvi struggles despite pre-training?

---
## References

1. **Szwarcman, D., et al. (2024).** Prithvi-EO-2.0: A Versatile Multi-Temporal Foundation Model for Earth Observation Applications. *IEEE JSTARS*. [arXiv:2412.02732](https://arxiv.org/abs/2412.02732)

2. **Jakubik, J., et al. (2023).** Foundation Models for Generalist Geospatial Artificial Intelligence. *ICLR 2024*. [arXiv:2310.18660](https://arxiv.org/abs/2310.18660)

3. **Hu, E. J., et al. (2022).** LoRA: Low-Rank Adaptation of Large Language Models. *ICLR 2022*. [arXiv:2106.09685](https://arxiv.org/abs/2106.09685)

4. **Lacoste, A., et al. (2023).** GEO-Bench: Toward Foundation Models for Earth Monitoring. *NeurIPS 2023 Datasets and Benchmarks*.

5. **HLS Burn Scars Dataset.** [huggingface.co/datasets/ibm-nasa-geospatial/hls_burn_scars](https://huggingface.co/datasets/ibm-nasa-geospatial/hls_burn_scars)

6. **Prithvi-EO-2.0 Models.** [huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-300M](https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-300M)

7. **TerraTorch.** [github.com/IBM/terratorch](https://github.com/IBM/terratorch)

8. **NASA-IMPACT Prithvi-EO-2.0 Repository.** [github.com/NASA-IMPACT/Prithvi-EO-2.0](https://github.com/NASA-IMPACT/Prithvi-EO-2.0)

9. **TerraTorch Examples.** [github.com/blumenstiel/TerraTorch-Examples](https://github.com/blumenstiel/TerraTorch-Examples)